This notebook benchmarks the CRYSTALS-Kyber KEM (Kyber512, Kyber768 and Kyber1024) on an **Alveo U50 FPGA** and compares it against the **C reference implementation running on the host CPU**. For every variant (512/768/1024), operation (`keygen`, `enc`, `dec`) and hardware version (`v1`, `v2`, `v3`) it measures time, clock cycles, power, throughput, energy and area-time product, and finally consolidates everything into a single table (displayed, and exported to Excel and CSV).

# Importing Libraries

**Step 1 – Imports.** The next cell loads:
- `ctypes` / `subprocess` / `os`: compile the C reference code into shared libraries and call it from Python.
- `numpy` / `pandas`: buffers for the cryptographic data and tabular results.
- `pyxrt`: the Xilinx Runtime (XRT) Python bindings, used to load `.xclbin` bitstreams and run kernels on the FPGA.
- `benchmark_utils` (local helper module): `get_clock_mhz` (reads the kernel clock from the xclbin), `get_cpu_mhz` (CPU frequency), `SensorSampler` (samples power in the background during a run) and `load_kat_seeds` (loads Known-Answer-Test seeds from a text file).
- `%autoreload 2` re-imports `benchmark_utils` automatically if you edit it, so you do not need to restart the kernel.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import gc
import re
import ctypes
import subprocess
import time

import numpy as np
import pandas as pd
import pyxrt

from IPython.display import display
from benchmark_utils import get_clock_mhz, get_cpu_mhz, SensorSampler, load_kat_seeds

# Global Configurations

**Step 2 – Global configuration.** The next cell defines everything that parametrizes the experiment:
- `KYBER_PARAMS`: for each Kyber variant, the module rank `K` and the sizes in bytes of the public key, secret key, ciphertext and shared secret, plus the number of random bytes ("coins") consumed by key generation (64) and encapsulation (32).
- `VERSIONS`: the three hardware design versions under test (`v1`, `v2`, `v3`).
- `get_hw_cfg(variant, op, version)`: returns the path of the `.xclbin` bitstream and the kernel name for a given variant/operation/version.
- `REF_DIR`, `KEYGEN_SEEDS_PATH`, `ENC_SEEDS_PATH`: location of the C reference code and of the KAT seed files used by the correctness check.
- `RUNS` (number of timed executions per configuration) and `DEVICE_INDEX` (which FPGA card to use).
- `KERNEL_LUTS`: the LUT usage of every kernel (taken from the synthesis/implementation reports). It is the "area" term used later to compute the area-time product (ATP = LUTs × time). Entries left as `None` have no data yet, and their ATP becomes `NaN`.

In [2]:
KYBER_PARAMS = {
    "Kyber512": {
        "K":              2,
        "pk_bytes":     800,
        "sk_bytes":    1632,
        "ct_bytes":     768,
        "ss_bytes":      32,
        "coins_kg":      64,   
        "coins_enc":     32
    },
    "Kyber768": {
        "K":              3,
        "pk_bytes":    1184,
        "sk_bytes":    2400,
        "ct_bytes":    1088,
        "ss_bytes":      32,
        "coins_kg":      64,
        "coins_enc":     32
    },
    "Kyber1024": {
        "K":              4,
        "pk_bytes":    1568,
        "sk_bytes":    3168,
        "ct_bytes":    1568,
        "ss_bytes":      32,
        "coins_kg":      64,
        "coins_enc":     32
    },
}

VERSIONS = ["v1", "v2", "v3"]

def get_hw_cfg(variant: str, op: str, version: str) -> dict:
    bits = variant.replace("Kyber", "")
    
    if op == "keygen":
        return {
            "xclbin": f"../keygen/kyber-keygen-{version}/kyber{bits}_keygen.xclbin",
            "kernel_name": "keygen"
        }
    elif op == "enc":
        return {
            "xclbin": f"../encaps/kyber-encaps-{version}/kyber{bits}_enc.xclbin",
            "kernel_name": "enc"
        }
    elif op == "dec":
        return {
            "xclbin": f"../decaps/kyber-decaps-{version}/kyber{bits}_dec.xclbin",
            "kernel_name": "dec"
        }

REF_DIR = "../../kyber/ref"
KEYGEN_SEEDS_PATH = "keygen_seeds.txt"
ENC_SEEDS_PATH = "enc_seeds.txt"
RUNS         = 1000
DEVICE_INDEX = 0 

OPERATIONS = ["keygen", "enc", "dec"]

KERNEL_LUTS = {
    (variant, op, version): None
    for variant in KYBER_PARAMS
    for op in OPERATIONS
    for version in VERSIONS
}

# Version 1 
KERNEL_LUTS[("Kyber512", "keygen", "v1")] = 35634
KERNEL_LUTS[("Kyber512", "enc",    "v1")] = 51349 
KERNEL_LUTS[("Kyber512", "dec",    "v1")] = 63478 

KERNEL_LUTS[("Kyber768", "keygen", "v1")] = 33776
KERNEL_LUTS[("Kyber768", "enc",    "v1")] = 58691 
KERNEL_LUTS[("Kyber768", "dec",    "v1")] = 71234

KERNEL_LUTS[("Kyber1024", "keygen", "v1")] = 33722
KERNEL_LUTS[("Kyber1024", "enc",    "v1")] = 58823
KERNEL_LUTS[("Kyber1024", "dec",    "v1")] = 71351

# Version 2
KERNEL_LUTS[("Kyber512", "keygen", "v2")] = 42104
KERNEL_LUTS[("Kyber512", "enc",    "v2")] = 67746 
KERNEL_LUTS[("Kyber512", "dec",    "v2")] = 79438

KERNEL_LUTS[("Kyber768", "keygen", "v2")] = 45553
KERNEL_LUTS[("Kyber768", "enc",    "v2")] = 71896
KERNEL_LUTS[("Kyber768", "dec",    "v2")] = 83817 

KERNEL_LUTS[("Kyber1024", "keygen", "v2")] = 50105
KERNEL_LUTS[("Kyber1024", "enc",    "v2")] = 78199
KERNEL_LUTS[("Kyber1024", "dec",    "v2")] = 90161

# Version 3 
KERNEL_LUTS[("Kyber512", "keygen", "v3")] = 54111
KERNEL_LUTS[("Kyber512", "enc",    "v3")] = 73276
KERNEL_LUTS[("Kyber512", "dec",    "v3")] = 67004

KERNEL_LUTS[("Kyber768", "keygen", "v3")] = 52271
KERNEL_LUTS[("Kyber768", "enc",    "v3")] = 73006
KERNEL_LUTS[("Kyber768", "dec",    "v3")] = 77207

KERNEL_LUTS[("Kyber1024", "keygen", "v3")] = 52583
KERNEL_LUTS[("Kyber1024", "enc",    "v3")] = 72725
KERNEL_LUTS[("Kyber1024", "dec",    "v3")] = 76741

**Step 3 – Statistics helper.** `calc_stats` receives a list of measurements and returns a dictionary with its **mean, min, max, sample standard deviation, standard error (SE), 95% confidence-interval half-width (CI = 1.96 × SE) and relative error (CI / mean × 100)**. Each key is prefixed (e.g. `time_total_mean`) so that several metrics can be merged into one result row. Empty inputs or `NaN` values are handled safely.

In [3]:
def calc_stats(data_list, prefix=""):
    if not data_list:
        return {f"{prefix}mean": np.nan, f"{prefix}max": np.nan, f"{prefix}min": np.nan, 
                f"{prefix}std": np.nan, f"{prefix}se": np.nan, f"{prefix}ci": np.nan, f"{prefix}rel_err": np.nan}
    
    arr = np.array(data_list)
    arr = arr[~np.isnan(arr)]
    
    if len(arr) == 0:
        return {f"{prefix}mean": np.nan, f"{prefix}max": np.nan, f"{prefix}min": np.nan, 
                f"{prefix}std": np.nan, f"{prefix}se": np.nan, f"{prefix}ci": np.nan, f"{prefix}rel_err": np.nan}
        
    n = len(arr)
    mean = np.mean(arr)
    std = np.std(arr, ddof=1) if n > 1 else 0.0
    se = std / np.sqrt(n) if n > 0 else 0.0
    ci = 1.96 * se
    
    rel_err = (ci / mean * 100) if mean > 0 else 0.0
    
    return {
        f"{prefix}mean": mean,
        f"{prefix}max": np.max(arr),
        f"{prefix}min": np.min(arr),
        f"{prefix}std": std,
        f"{prefix}se": se,
        f"{prefix}ci": ci,
        f"{prefix}rel_err": rel_err
    }

# Compilation of Reference Libraries (Software)

The next cell:
1. Writes a **dummy `randombytes()`** implementation. The reference code normally pulls entropy from the OS, but the benchmark uses the deterministic `*_derand` entry points (coins are passed explicitly), so the symbol only needs to exist to link successfully.
2. Defines `compile_kyber_lib`, which runs `gcc -O3 -shared -fPIC` on the Kyber reference sources with `-DKYBER_K=<K>` to build one shared library per variant (`kyber512_ref.so`, `kyber768_ref.so`, `kyber1024_ref.so`), loads it with `ctypes.CDLL` and declares the C signatures (argument and return types) of key generation, encapsulation and decapsulation.
3. Compiles the three variants and stores the loaded libraries in `SW_LIBS`.

In [4]:
# Dummy random bytes (just for compilation, not used)
dummy_rb_path = os.path.join(REF_DIR, "dummy_randombytes.c")
with open(dummy_rb_path, "w") as f:
    f.write("#include <stddef.h>\n#include <stdint.h>\nvoid randombytes(uint8_t *out, size_t outlen) {}\n")

_SOURCES = [
    "kem.c", "indcpa.c", "polyvec.c", "poly.c",
    "ntt.c", "cbd.c", "reduce.c", "verify.c",
    "fips202.c", "symmetric-shake.c",
    "dummy_randombytes.c" 
]

def compile_kyber_lib(variant: str, params: dict) -> ctypes.CDLL:
    k       = params["K"]
    so_path = os.path.abspath(f"kyber{k*256}_ref.so")
    srcs    = " ".join(os.path.join(REF_DIR, s) for s in _SOURCES)

    cmd = (
        f"gcc -O3 -shared -fPIC "
        f"-I{REF_DIR} "
        f"-DKYBER_K={k} "
        f"'-DKYBER_NAMESPACE(s)=s' "
        f"{srcs} "
        f"-o {so_path}"
    )
    ret = os.system(cmd)
    if ret != 0:
        raise RuntimeError(f"Failed to compile for Kyber{variant}")

    lib = ctypes.CDLL(so_path)

    # Keygen
    fn_kg = lib.keypair_derand
    fn_kg.restype  = ctypes.c_int
    fn_kg.argtypes = [
        ctypes.POINTER(ctypes.c_uint8),   # pk
        ctypes.POINTER(ctypes.c_uint8),   # sk
        ctypes.POINTER(ctypes.c_uint8),   # coins
    ]
    lib.crypto_kem_keypair_derand = fn_kg

    # Encapsulation
    fn_enc = lib.enc_derand
    fn_enc.restype  = ctypes.c_int
    fn_enc.argtypes = [
        ctypes.POINTER(ctypes.c_uint8),   # ct
        ctypes.POINTER(ctypes.c_uint8),   # ss
        ctypes.POINTER(ctypes.c_uint8),   # pk
        ctypes.POINTER(ctypes.c_uint8),   # coins
    ]
    lib.crypto_kem_enc_derand = fn_enc

    # Decapsulation
    fn_dec = lib.dec
    fn_dec.restype  = ctypes.c_int
    fn_dec.argtypes = [
        ctypes.POINTER(ctypes.c_uint8),   # ss (output)
        ctypes.POINTER(ctypes.c_uint8),   # ct (input)
        ctypes.POINTER(ctypes.c_uint8),   # sk (input)
    ]
    lib.crypto_kem_dec = fn_dec

    return lib

SW_LIBS = {}
for variant, params in KYBER_PARAMS.items():
    print(f"Compiling {variant}", end=" ")
    SW_LIBS[variant] = compile_kyber_lib(variant, params)
    print("OK")

Compiling Kyber512 OK
Compiling Kyber768 OK
Compiling Kyber1024 OK


# Generation Input

The next cell creates the **inputs for each operation**:
- `random_coins(n)` returns `n` random bytes from `os.urandom` as a `uint8` NumPy array, and `ptr()` converts a NumPy array into a C pointer for `ctypes`.
- `generate_keygen_input`: only the 64 random bytes needed by key generation.
- `generate_enc_input`: first generates a key pair with the software library, then returns the public key plus 32 fresh coins for encapsulation.
- `generate_dec_input`: generates a key pair, encapsulates with the software library, and returns the ciphertext, the secret key and the **expected shared secret**.

All three accept optional fixed coins (`coins_kg`, `coins_enc`), which is how KAT seeds are injected in the correctness check.

In [5]:
def random_coins(n: int) -> np.ndarray:
    return np.frombuffer(os.urandom(n), dtype=np.uint8)

def ptr(arr: np.ndarray) -> ctypes.POINTER:
    return arr.ctypes.data_as(ctypes.POINTER(ctypes.c_uint8))

def generate_keygen_input(params: dict) -> dict:
    return {"coins": random_coins(params["coins_kg"])}

def generate_enc_input(lib, params: dict, coins_kg: np.ndarray = None, coins_enc: np.ndarray = None) -> dict:
    pk = np.zeros(params["pk_bytes"], dtype=np.uint8)
    sk = np.zeros(params["sk_bytes"], dtype=np.uint8)
    if coins_kg is None:
        coins_kg = random_coins(params["coins_kg"])
    lib.crypto_kem_keypair_derand(ptr(pk), ptr(sk), ptr(coins_kg))

    if coins_enc is None:
        coins_enc = random_coins(params["coins_enc"])

    return {"pk": pk, "coins": coins_enc}

def generate_dec_input(lib, params: dict, coins_kg: np.ndarray = None, coins_enc: np.ndarray = None) -> dict:
    pk = np.zeros(params["pk_bytes"], dtype=np.uint8)
    sk = np.zeros(params["sk_bytes"], dtype=np.uint8)
    if coins_kg is None:
        coins_kg = random_coins(params["coins_kg"])
    lib.crypto_kem_keypair_derand(ptr(pk), ptr(sk), ptr(coins_kg))

    if coins_enc is None:
        coins_enc = random_coins(params["coins_enc"])
    ct = np.zeros(params["ct_bytes"], dtype=np.uint8)
    ss_expected = np.zeros(params["ss_bytes"], dtype=np.uint8)
    lib.crypto_kem_enc_derand(ptr(ct), ptr(ss_expected), ptr(pk), ptr(coins_enc))

    return {"ct": ct, "sk": sk, "expected_ss": ss_expected}

# Benchmark Software

The next cell implements the **software (CPU) benchmark**:
- `run_sw_operation_batch` pre-allocates all output buffers and C pointers (so allocation is not timed), then runs the selected operation `runs` times. Each call is timed with `time.perf_counter()`; the time is converted to **cycles** with `cycles = time_ms × CPU_MHz × 1000`. A run counts as correct when the C function returns 0.
- `benchmark_software` loops over every variant and operation, builds `RUNS` independent inputs, starts the `SensorSampler` (CPU power), executes the batch, stops the sampler and computes the statistics. It also derives **throughput** (1000 / mean ms), **energy per operation** (mean power × mean time, in µJ), and returns a DataFrame with one row per (variant, operation). For software there is no host/device transfer, so the "total" and "exec" metrics are identical.

In [6]:
def run_sw_operation_batch(lib, operation: str, inputs_list: list, params: dict, cpu_mhz: float):
    runs = len(inputs_list)
    correct_runs = 0
    times_ms = []
    cycles = []

    if operation == "keygen":
        pks = [np.zeros(params["pk_bytes"], dtype=np.uint8) for _ in range(runs)]
        sks = [np.zeros(params["sk_bytes"], dtype=np.uint8) for _ in range(runs)]
        ptr_coins = [ptr(inp["coins"]) for inp in inputs_list]
        ptr_pks = [ptr(pk) for pk in pks]
        ptr_sks = [ptr(sk) for sk in sks]

        for i in range(runs):
            t0 = time.perf_counter()
            ret = lib.crypto_kem_keypair_derand(ptr_pks[i], ptr_sks[i], ptr_coins[i])
            t1 = time.perf_counter()
            if ret == 0: correct_runs += 1
            
            elapsed_ms = (t1 - t0) * 1_000
            times_ms.append(elapsed_ms)
            cycles.append(int(elapsed_ms * cpu_mhz * 1_000))

    elif operation == "enc":
        cts = [np.zeros(params["ct_bytes"], dtype=np.uint8) for _ in range(runs)]
        sss = [np.zeros(params["ss_bytes"], dtype=np.uint8) for _ in range(runs)]
        ptr_pks = [ptr(inp["pk"]) for inp in inputs_list]
        ptr_coins = [ptr(inp["coins"]) for inp in inputs_list]
        ptr_cts = [ptr(ct) for ct in cts]
        ptr_sss = [ptr(ss) for ss in sss]

        for i in range(runs):
            t0 = time.perf_counter()
            ret = lib.crypto_kem_enc_derand(ptr_cts[i], ptr_sss[i], ptr_pks[i], ptr_coins[i])
            t1 = time.perf_counter()
            if ret == 0: correct_runs += 1
            
            elapsed_ms = (t1 - t0) * 1_000
            times_ms.append(elapsed_ms)
            cycles.append(int(elapsed_ms * cpu_mhz * 1_000))

    elif operation == "dec":
        sss = [np.zeros(params["ss_bytes"], dtype=np.uint8) for _ in range(runs)]
        ptr_cts = [ptr(inp["ct"]) for inp in inputs_list]
        ptr_sks = [ptr(inp["sk"]) for inp in inputs_list]
        ptr_sss = [ptr(ss) for ss in sss]

        for i in range(runs):
            t0 = time.perf_counter()
            ret = lib.crypto_kem_dec(ptr_sss[i], ptr_cts[i], ptr_sks[i])
            t1 = time.perf_counter()
            if ret == 0: correct_runs += 1
            
            elapsed_ms = (t1 - t0) * 1_000
            times_ms.append(elapsed_ms)
            cycles.append(int(elapsed_ms * cpu_mhz * 1_000))

    return times_ms, cycles, correct_runs == runs
    
def benchmark_software(runs: int = RUNS) -> pd.DataFrame:
    cpu_mhz = get_cpu_mhz()
    print(f"  CPU Clock Base/Max: {cpu_mhz:.1f} MHz\n")
    records = []

    for variant, params in KYBER_PARAMS.items():
        print(f"  Variant (SW): {variant}")
        lib = SW_LIBS[variant]

        for op in OPERATIONS:
            inputs_list = []
            for _ in range(runs):
                if op == "keygen": inputs_list.append(generate_keygen_input(params))
                elif op == "enc": inputs_list.append(generate_enc_input(lib, params))
                elif op == "dec": inputs_list.append(generate_dec_input(lib, params))

            sampler = SensorSampler("software")
            sampler.start()

            times_ms, cycles, ok = run_sw_operation_batch(lib, op, inputs_list, params, cpu_mhz)

            _, powers = sampler.stop()

            time_total_stats = calc_stats(times_ms, "time_total_")
            time_exec_stats  = calc_stats(times_ms, "time_exec_")
            cyc_total_stats  = calc_stats(cycles,   "cyc_total_")
            cyc_exec_stats   = calc_stats(cycles,   "cyc_exec_")
            pwr_stats        = calc_stats(powers,   "pwr_")

            status = "OK" if ok else "ERROR"
            print(f"  {op:10s}: {time_total_stats['time_total_mean']:.4f} ms "
                  f"(±{time_total_stats['time_total_ci']:.4f}) | "
                  f"{cyc_total_stats['cyc_total_mean']:,.0f} cycles | {status}")

            mean_ms = time_total_stats["time_total_mean"]
            throughput_ops_s = (1000.0 / mean_ms) if mean_ms > 0 else float("nan")
            energy_uj = (pwr_stats["pwr_mean"] * mean_ms * 1_000.0) if not np.isnan(pwr_stats["pwr_mean"]) else float("nan")

            record = {
                "version": "SW", "variant": variant, "operation": op,
                "correct": ok, "runs": runs,
                "platform": "software",
                "clock_mhz": cpu_mhz,
                "throughput_total_ops_s": throughput_ops_s,
                "throughput_exec_ops_s":  throughput_ops_s,
                "energy_total_uj": energy_uj,
                "energy_exec_uj":  energy_uj,
            }
            for d in (time_total_stats, time_exec_stats, cyc_total_stats, cyc_exec_stats, pwr_stats):
                record.update(d)
            records.append(record)

    return pd.DataFrame(records)


# Benchmark Hardware

The next cell contains the FPGA benchmark. It is organized in layers:
- **Buffer helpers**: `alloc_bo` allocates an XRT buffer object (BO), writes the host data and syncs it to the device (used by the correctness check). `create_bo` allocates and fills the host-side buffer *without* syncing, so the host→device transfer can be included inside the timed region. `sync_to_device` and `read_bo` perform the host→device and device→host transfers.
- **`timed_hw_run`**: executes the kernel once and records two time bases: **total** = H2D transfer + kernel launch + execution + D2H transfer, and **exec** = kernel execution only (from just after the launch until `run.wait()` returns). Both are also converted to cycles using the kernel clock.
- **`run_hw_keygen` / `run_hw_enc` / `run_hw_dec`**: create the BOs in the argument order expected by each kernel and call `timed_hw_run`. They do not compare results against the software reference; correctness is checked separately in the *Correctness Check* section below.
- **`benchmark_hardware`**: for every version, variant and operation it loads the xclbin, creates the kernel, reads the kernel clock, executes one **warm-up run** (discarded), generates `RUNS` inputs, then runs and times them while the `SensorSampler` records power. It computes statistics, throughput, energy and the **ATP** (kernel LUTs × mean time) for both time bases, and releases the device resources before moving on. Missing xclbin files are skipped with a message. Note that the power mean is measured over the whole batch, so the energy for the "exec" base is an approximation.

In [7]:
def alloc_bo(device, kernel, data: np.ndarray, group_idx: int) -> pyxrt.bo:
    """Allocate BO, write host buffer AND sync to device (used by the correctness check)."""
    bo = pyxrt.bo(device, data.nbytes, pyxrt.bo.normal, kernel.group_id(group_idx))
    bo.write(bytes(data), 0)
    bo.sync(pyxrt.xclBOSyncDirection.XCL_BO_SYNC_BO_TO_DEVICE, data.nbytes, 0)
    return bo


def create_bo(device, kernel, data: np.ndarray, group_idx: int) -> pyxrt.bo:
    """Allocate BO and fill the host-side buffer WITHOUT syncing to the device.
    The H2D transfer is done later, inside the timed region."""
    bo = pyxrt.bo(device, data.nbytes, pyxrt.bo.normal, kernel.group_id(group_idx))
    bo.write(bytes(data), 0)
    return bo


def sync_to_device(bo: pyxrt.bo, n_bytes: int) -> None:
    """Host -> device transfer."""
    bo.sync(pyxrt.xclBOSyncDirection.XCL_BO_SYNC_BO_TO_DEVICE, n_bytes, 0)


def read_bo(bo: pyxrt.bo, n_bytes: int) -> np.ndarray:
    """Device -> host transfer followed by the host-side read."""
    bo.sync(pyxrt.xclBOSyncDirection.XCL_BO_SYNC_BO_FROM_DEVICE, n_bytes, 0)
    return np.frombuffer(bo.read(n_bytes, 0), dtype=np.uint8).copy()


def timed_hw_run(kernel, in_bos, out_bos, args, clock_mhz: float) -> dict:
    """
    Run the kernel once and measure two time bases:
      - total : H2D transfer + kernel launch + execution + D2H transfer
      - exec  : kernel execution only (right after launch until wait() returns)

    in_bos  : list of (bo, n_bytes) transferred host -> device
    out_bos : list of (bo, n_bytes) transferred device -> host
    args    : BOs in the exact order expected by the kernel call
    """
    t0 = time.perf_counter()
    for bo, n in in_bos:                    # H2D transfer
        sync_to_device(bo, n)

    run = kernel(*args)                     # kernel launch (asynchronous)
    t1 = time.perf_counter()
    run.wait()                              # wait for kernel completion
    t2 = time.perf_counter()

    outputs = [read_bo(bo, n) for bo, n in out_bos]   # D2H transfer + host read
    t3 = time.perf_counter()

    total_ms = (t3 - t0) * 1_000            # H2D + launch + execution + D2H
    exec_ms  = (t2 - t1) * 1_000            # execution only

    return {
        "time_total_ms": total_ms,
        "time_exec_ms":  exec_ms,
        "cyc_total":     int(total_ms * clock_mhz * 1_000),
        "cyc_exec":      int(exec_ms  * clock_mhz * 1_000),
        "ok":            True,
        "outputs":       outputs,
    }


def run_hw_keygen(device, kernel, inputs: dict, params: dict, clock_mhz: float) -> dict:
    coins  = inputs["coins"]
    pk_out = np.zeros(params["pk_bytes"], dtype=np.uint8)
    sk_out = np.zeros(params["sk_bytes"], dtype=np.uint8)

    bo_coins = create_bo(device, kernel, coins,  0)
    bo_pk    = create_bo(device, kernel, pk_out, 1)
    bo_sk    = create_bo(device, kernel, sk_out, 2)

    return timed_hw_run(
        kernel,
        in_bos    = [(bo_coins, coins.nbytes)],
        out_bos   = [(bo_pk, pk_out.nbytes), (bo_sk, sk_out.nbytes)],
        args      = (bo_coins, bo_pk, bo_sk),
        clock_mhz = clock_mhz,
    )


def run_hw_enc(device, kernel, inputs: dict, params: dict, clock_mhz: float) -> dict:
    pk     = inputs["pk"]
    coins  = inputs["coins"]
    ct_out = np.zeros(params["ct_bytes"], dtype=np.uint8)
    ss_out = np.zeros(params["ss_bytes"], dtype=np.uint8)

    bo_pk    = create_bo(device, kernel, pk,     0)
    bo_coins = create_bo(device, kernel, coins,  1)
    bo_ct    = create_bo(device, kernel, ct_out, 2)
    bo_ss    = create_bo(device, kernel, ss_out, 3)

    return timed_hw_run(
        kernel,
        in_bos    = [(bo_pk, pk.nbytes), (bo_coins, coins.nbytes)],
        out_bos   = [(bo_ct, ct_out.nbytes), (bo_ss, ss_out.nbytes)],
        args      = (bo_pk, bo_coins, bo_ct, bo_ss),
        clock_mhz = clock_mhz,
    )


def run_hw_dec(device, kernel, inputs: dict, params: dict, clock_mhz: float) -> dict:
    ct     = inputs["ct"]
    sk     = inputs["sk"]
    ss_out = np.zeros(params["ss_bytes"], dtype=np.uint8)

    bo_ss = create_bo(device, kernel, ss_out, 0)
    bo_ct = create_bo(device, kernel, ct,     1)
    bo_sk = create_bo(device, kernel, sk,     2)

    return timed_hw_run(
        kernel,
        in_bos    = [(bo_ct, ct.nbytes), (bo_sk, sk.nbytes)],
        out_bos   = [(bo_ss, ss_out.nbytes)],
        args      = (bo_ss, bo_ct, bo_sk),
        clock_mhz = clock_mhz,
    )

_HW_RUNNERS = {
    "keygen": run_hw_keygen,
    "enc":    run_hw_enc,
    "dec":    run_hw_dec
}

def benchmark_hardware(runs: int = RUNS,
                        device_index: int = DEVICE_INDEX) -> pd.DataFrame:
    records = []

    for version in VERSIONS:
        print(f"\nHardware Version: {version}")
        for variant, params in KYBER_PARAMS.items():
            print(f"  Variant (HW) [{version}]: {variant}")

            lib = SW_LIBS[variant]

            for op in OPERATIONS:
                cfg        = get_hw_cfg(variant, op, version)

                if not os.path.isfile(cfg["xclbin"]):
                    print(f"  {op:10s}: File not found: {cfg['xclbin']}")
                    continue

                device     = pyxrt.device(device_index)
                xclbin_obj = pyxrt.xclbin(cfg["xclbin"])
                device.load_xclbin(xclbin_obj)
                uuid       = device.get_xclbin_uuid()
                kernel     = pyxrt.kernel(device, uuid, cfg["kernel_name"])
                clock_mhz  = get_clock_mhz(cfg["xclbin"])

                print(f"  {op:10s}: xclbin loaded | clock = {clock_mhz:.1f} MHz")

                # Warm-up run (result discarded)
                if op == "keygen":
                    wu_inputs = generate_keygen_input(params)
                elif op == "enc":
                    wu_inputs = generate_enc_input(lib, params)
                elif op == "dec":
                    wu_inputs = generate_dec_input(lib, params)

                _HW_RUNNERS[op](device, kernel, wu_inputs, params, clock_mhz)

                metrics = {k: [] for k in ("time_total_ms", "time_exec_ms", "cyc_total", "cyc_exec")}
                correct_runs = 0

                inputs_list = []
                for _ in range(runs):
                    if op == "keygen":
                        inputs_list.append(generate_keygen_input(params))
                    elif op == "enc":
                        inputs_list.append(generate_enc_input(lib, params))
                    elif op == "dec":
                        inputs_list.append(generate_dec_input(lib, params))

                sampler = SensorSampler("hardware")
                sampler.start()

                for inputs in inputs_list:
                    r = _HW_RUNNERS[op](device, kernel, inputs, params, clock_mhz)
                    for k in metrics:
                        metrics[k].append(r[k])
                    if r["ok"]:
                        correct_runs += 1

                _, powers = sampler.stop()

                time_total_stats = calc_stats(metrics["time_total_ms"], "time_total_")
                time_exec_stats  = calc_stats(metrics["time_exec_ms"],  "time_exec_")
                cyc_total_stats  = calc_stats(metrics["cyc_total"],     "cyc_total_")
                cyc_exec_stats   = calc_stats(metrics["cyc_exec"],      "cyc_exec_")
                pwr_stats        = calc_stats(powers, "pwr_")
    
                status  = "OK" if correct_runs == runs else f"ERROR ({correct_runs}/{runs})"

                print(f"           → total: {time_total_stats['time_total_mean']:.4f} ms "
                      f"(±{time_total_stats['time_total_ci']:.4f}) | "
                      f"{cyc_total_stats['cyc_total_mean']:,.0f} cycles")
                print(f"           → exec : {time_exec_stats['time_exec_mean']:.4f} ms "
                      f"(±{time_exec_stats['time_exec_ci']:.4f}) | "
                      f"{cyc_exec_stats['cyc_exec_mean']:,.0f} cycles | {status}")

                # Derived metrics, one per time base
                mean_total_ms = time_total_stats["time_total_mean"]
                mean_exec_ms  = time_exec_stats["time_exec_mean"]

                def _throughput(ms):
                    return (1000.0 / ms) if ms > 0 else float("nan")

                def _energy_uj(ms):
                    # NOTE: the power mean is sampled over the whole batch, so it is the same
                    # for both bases; "exec" energy is therefore an approximation.
                    return (pwr_stats["pwr_mean"] * ms * 1_000.0) if not np.isnan(pwr_stats["pwr_mean"]) else float("nan")

                throughput_total_ops_s = _throughput(mean_total_ms)
                throughput_exec_ops_s  = _throughput(mean_exec_ms)
                energy_total_uj        = _energy_uj(mean_total_ms)
                energy_exec_uj         = _energy_uj(mean_exec_ms)

                kernel_luts       = KERNEL_LUTS.get((variant, op, version))
                atp_total_luts_ms = (kernel_luts * mean_total_ms) if kernel_luts is not None else float("nan")
                atp_exec_luts_ms  = (kernel_luts * mean_exec_ms)  if kernel_luts is not None else float("nan")

                record = {
                    "version": version, "variant": variant, "operation": op,
                    "correct": correct_runs == runs, "runs": runs,
                    "platform": "hardware",
                    "clock_mhz": clock_mhz,
                    "throughput_total_ops_s": throughput_total_ops_s,
                    "throughput_exec_ops_s":  throughput_exec_ops_s,
                    "energy_total_uj": energy_total_uj,
                    "energy_exec_uj":  energy_exec_uj,
                    "kernel_luts": kernel_luts,
                    "atp_total_luts_ms": atp_total_luts_ms,
                    "atp_exec_luts_ms":  atp_exec_luts_ms,
                }
                for d in (time_total_stats, time_exec_stats, cyc_total_stats, cyc_exec_stats, pwr_stats):
                    record.update(d)
                records.append(record)

                del kernel
                del xclbin_obj
                del device
                gc.collect()

    return pd.DataFrame(records)


**Run the benchmarks.** The next cell executes the software benchmark (`sw_df`) and the hardware benchmark (`hw_df`). This is the longest-running cell: it performs `RUNS` executions for every variant, operation and version, and prints the mean time (± 95% CI), cycles and status as it progresses.

In [9]:
print("BENCHMARK - SOFTWARE (CPU)")
sw_df = benchmark_software(runs=RUNS)

print("BENCHMARK - HARDWARE (FPGA)")
hw_df = benchmark_hardware(runs=RUNS, device_index=DEVICE_INDEX)

BENCHMARK - SOFTWARE (CPU)
  CPU Clock Base/Max: 4400.0 MHz

  Variant (SW): Kyber512
  keygen    : 0.0285 ms (±0.0001) | 125,305 cycles | OK
  enc       : 0.0359 ms (±0.0001) | 158,034 cycles | OK
  dec       : 0.0469 ms (±0.0000) | 206,233 cycles | OK
  Variant (SW): Kyber768
  keygen    : 0.0469 ms (±0.0001) | 206,213 cycles | OK
  enc       : 0.0570 ms (±0.0003) | 250,949 cycles | OK
  dec       : 0.0710 ms (±0.0000) | 312,411 cycles | OK
  Variant (SW): Kyber1024
  keygen    : 0.0767 ms (±0.0001) | 337,379 cycles | OK
  enc       : 0.0905 ms (±0.0001) | 398,027 cycles | OK
  dec       : 0.1096 ms (±0.0001) | 482,081 cycles | OK
BENCHMARK - HARDWARE (FPGA)

Hardware Version: v1
  Variant (HW) [v1]: Kyber512
  keygen    : xclbin loaded | clock = 500.0 MHz
           → total: 0.2344 ms (±0.0003) | 117,214 cycles
           → exec : 0.1814 ms (±0.0002) | 90,701 cycles | OK
  enc       : xclbin loaded | clock = 500.0 MHz
           → total: 0.3796 ms (±0.0003) | 189,799 cycles
        

# Consolidated Results Table

The next cell merges `hw_df` and `sw_df` into `combined` and builds a single, readable table with one row per (Variant, Operation, Version) and grouped columns (via a `MultiIndex`):
- **hardware**: frequency, time (total and exec), cycles (total and exec), power, throughput, energy and ATP. Statistic columns are shown as *Mean ± CI*, *Min-Max* and *Rel Err %*.
- **software**: the same metrics for the CPU reference implementation.
- **speedup**: comparisons between the software baseline and each hardware version, for cycles, time, throughput (HW/SW) and energy (SW/HW). Values above 1× mean the hardware is better.

The table is displayed at the end of the cell.

In [11]:
combined = pd.concat([hw_df, sw_df], ignore_index=True)

def add_stats_block(row, group, stats, prefix, fmt):
    """Adds the 'Mean ± CI', 'Min-Max' and 'Rel Err %' columns for one metric."""
    row[(group, "Mean ± CI")] = f"{stats[prefix + 'mean']:{fmt}} ± {stats[prefix + 'ci']:{fmt}}"
    row[(group, "Min-Max")]   = f"{stats[prefix + 'min']:{fmt}} - {stats[prefix + 'max']:{fmt}}"
    row[(group, "Rel Err %")] = f"{stats[prefix + 'rel_err']:.2f}%"

def safe_ratio(num, den):
    return num / den if den and not np.isnan(den) and not np.isnan(num) else float("nan")

rows = []
for variant in KYBER_PARAMS.keys():
    for op in OPERATIONS:
        sw_sub = combined[(combined["variant"] == variant) & (combined["operation"] == op) & (combined["platform"] == "software")]

        for version in VERSIONS:
            hw_sub = combined[(combined["variant"] == variant) & (combined["operation"] == op) & (combined["platform"] == "hardware") & (combined["version"] == version)]

            if hw_sub.empty:
                continue

            row = {
                ("Info", "Variant"): variant,
                ("Info", "Operation"): op,
                ("Info", "Version"): version
            }

            h_s = hw_sub.iloc[0]
            # Hardware - Frequency
            row[("hardware", "Freq (MHz)")] = f"{h_s['clock_mhz']:.1f}"

            # Hardware - Time (total = H2D + execution + D2H | exec = kernel only)
            add_stats_block(row, "hardware_Time total (ms)", h_s, "time_total_", ".4f")
            add_stats_block(row, "hardware_Time exec (ms)",  h_s, "time_exec_",  ".4f")

            # Hardware - Cycles
            add_stats_block(row, "hardware_Cycles total", h_s, "cyc_total_", ",.0f")
            add_stats_block(row, "hardware_Cycles exec",  h_s, "cyc_exec_",  ",.0f")

            # Hardware - Power
            add_stats_block(row, "hardware_Power (W)", h_s, "pwr_", ".2f")

            # Hardware - Throughput
            row[("hardware_Throughput", "total (ops/s)")] = f"{h_s['throughput_total_ops_s']:.2f}"
            row[("hardware_Throughput", "exec (ops/s)")]  = f"{h_s['throughput_exec_ops_s']:.2f}"

            # Hardware - Energy
            row[("hardware_Energy", "total (μJ/op)")] = f"{h_s['energy_total_uj']:.2f}"
            row[("hardware_Energy", "exec (μJ/op)")]  = f"{h_s['energy_exec_uj']:.2f}"

            # Hardware - ATP (Area-Time Product)
            for label, key in (("total (LUTs × ms)", "atp_total_luts_ms"), ("exec (LUTs × ms)", "atp_exec_luts_ms")):
                row[("hardware_ATP", label)] = f"{h_s[key]:,.2f}" if pd.notna(h_s[key]) else "—"

            if not sw_sub.empty:
                s_s = sw_sub.iloc[0]
                # Software - Frequency
                row[("software", "Freq (MHz)")] = f"{s_s['clock_mhz']:.1f}"

                # Software - Time
                add_stats_block(row, "software_Time (ms)", s_s, "time_total_", ".4f")

                # Software - Cycles
                add_stats_block(row, "software_Cycles", s_s, "cyc_total_", ",.0f")

                # Software - Power
                add_stats_block(row, "software_Power (W)", s_s, "pwr_", ".4f")

                # Software - Throughput
                row[("software_Throughput", "ops/s")] = f"{s_s['throughput_total_ops_s']:.2f}"

                # Software - Energy
                row[("software_Energy", "μJ/op")] = f"{s_s['energy_total_uj']:.6f}"

                # Speedups (SW baseline vs. HW with each time base)
                row[("speedup", "HW/SW cycles (total)")] = f"{safe_ratio(s_s['cyc_total_mean'], h_s['cyc_total_mean']):.2f}×"
                row[("speedup", "HW/SW cycles (exec)")]  = f"{safe_ratio(s_s['cyc_total_mean'], h_s['cyc_exec_mean']):.2f}×"

                row[("speedup", "HW/SW time (total)")] = f"{safe_ratio(s_s['time_total_mean'], h_s['time_total_mean']):.2f}×"
                row[("speedup", "HW/SW time (exec)")]  = f"{safe_ratio(s_s['time_total_mean'], h_s['time_exec_mean']):.2f}×"

                row[("speedup", "HW/SW throughput (total)")] = f"{safe_ratio(h_s['throughput_total_ops_s'], s_s['throughput_total_ops_s']):.2f}×"
                row[("speedup", "HW/SW throughput (exec)")]  = f"{safe_ratio(h_s['throughput_exec_ops_s'],  s_s['throughput_total_ops_s']):.2f}×"

                row[("speedup", "SW/HW energy (total)")] = f"{safe_ratio(s_s['energy_total_uj'], h_s['energy_total_uj']):.2f}×"
                row[("speedup", "SW/HW energy (exec)")]  = f"{safe_ratio(s_s['energy_total_uj'], h_s['energy_exec_uj']):.2f}×"

            rows.append(row)

table = pd.DataFrame(rows)
table.columns = pd.MultiIndex.from_tuples(table.columns)
table = table.set_index([("Info", "Variant"), ("Info", "Operation"), ("Info", "Version")])
table.index.names = ["Variant", "Operation", "Version"]

display(table)


hardware hardware_Time total (ms)  \
                            Freq (MHz)                Mean ± CI   
Variant   Operation Version                                       
Kyber512  keygen    v1           500.0          0.2344 ± 0.0003   
                    v2           500.0          0.1270 ± 0.0003   
                    v3           500.0          0.1181 ± 0.0005   
          enc       v1           500.0          0.3796 ± 0.0003   
                    v2           500.0          0.1603 ± 0.0006   
                    v3           500.0          0.1600 ± 0.0003   
          dec       v1           500.0          0.5223 ± 0.0007   
                    v2           500.0          0.1904 ± 0.0003   
                    v3           500.0          0.1901 ± 0.0003   
Kyber768  keygen    v1           500.0          0.3860 ± 0.0009   
                    v2           500.0          0.1696 ± 0.0004   
                    v3           500.0          0.1635 ± 0.0004   
          enc       v1           500.0          0.5535 ± 0.0013   
                    v2           500.0          0.2166 ± 0.0006   
                    v3           500.0          0.2023 ± 0.0004   
          dec       v1           500.0          0.7471 ± 0.0009   
                    v2           500.0          0.2556 ± 0.0005   
                    v3           500.0          0.3117 ± 0.0009   
Kyber1024 keygen    v1           500.0          0.5333 ± 0.0010   
                    v2           500.0          0.2392 ± 0.0004   
                    v3           500.0          0.2191 ± 0.0003   
          enc       v1           500.0          0.7296 ± 0.0015   
                    v2           500.0          0.2750 ± 0.0003   
                    v3           500.0          0.3143 ± 0.0004   
          dec       v1           500.0          0.9849 ± 0.0015   
                    v2           500.0          0.3364 ± 0.0004   
                    v3           500.0          0.3116 ± 0.0006   

                                                        \
                                     Min-Max Rel Err %   
Variant   Operation Version                              
Kyber512  keygen    v1       0.2266 - 0.3244     0.13%   
                    v2       0.1218 - 0.2531     0.25%   
                    v3       0.1136 - 0.2395     0.39%   
          enc       v1       0.3734 - 0.4721     0.09%   
                    v2       0.1546 - 0.2736     0.35%   
                    v3       0.1552 - 0.2415     0.18%   
          dec       v1       0.5170 - 0.7534     0.12%   
                    v2       0.1866 - 0.2898     0.18%   
                    v3       0.1870 - 0.2538     0.13%   
Kyber768  keygen    v1       0.3731 - 0.6201     0.24%   
                    v2       0.1640 - 0.2541     0.22%   
                    v3       0.1589 - 0.2693     0.23%   
          enc       v1       0.5330 - 0.6753     0.23%   
                    v2       0.2119 - 0.3282     0.27%   
                    v3       0.1984 - 0.3204     0.21%   
          dec       v1       0.7296 - 0.9187     0.13%   
                    v2       0.2478 - 0.3417     0.21%   
                    v3       0.3020 - 0.5946     0.30%   
Kyber1024 keygen    v1       0.5200 - 0.6815     0.18%   
                    v2       0.2333 - 0.3854     0.17%   
                    v3       0.2104 - 0.2609     0.12%   
          enc       v1       0.7118 - 0.9912     0.21%   
                    v2       0.2716 - 0.3952     0.12%   
                    v3       0.3075 - 0.4320     0.12%   
          dec       v1       0.9670 - 1.1541     0.15%   
                    v2       0.3287 - 0.4473     0.11%   
                    v3       0.3058 - 0.4437     0.18%   

                            hardware_Time exec (ms)                   \
                                          Mean ± CI          Min-Max   
Variant   Operation Version                                            
Kyber512  keygen    v1              0.1814 ± 0.0002  0.1744 - 

# Correctness Check (Known-Answer Tests)

The timed runs above only measure performance. In this section the hardware outputs are compared **byte by byte** against the software reference.

**Step 1 – Load the KAT seeds.** The next cell reads the key-generation seeds (64 bytes each) and the encapsulation seeds (32 bytes each) from the text files, so that software and hardware receive exactly the same deterministic inputs.

In [12]:
KEYGEN_SEEDS = load_kat_seeds(KEYGEN_SEEDS_PATH, seed_len=64)
print(f"Loaded {len(KEYGEN_SEEDS)} KAT seeds ({KEYGEN_SEEDS_PATH})")

ENC_SEEDS = load_kat_seeds(ENC_SEEDS_PATH, seed_len=32)
print(f"Loaded {len(ENC_SEEDS)} KAT seeds ({ENC_SEEDS_PATH})")

Loaded 100 KAT seeds (keygen_seeds.txt)
Loaded 100 KAT seeds (enc_seeds.txt)


**Step 2 – Compare hardware against software.** For every version and variant, the next cell loads each xclbin and, for up to 100 KAT vectors, runs the operation on both the FPGA and the C reference:
- **Keygen**: the hardware public key and secret key must match the reference exactly.
- **Encaps**: the hardware ciphertext and shared secret must match the reference exactly.
- **Decaps**: the hardware shared secret must match the expected one produced by the reference encapsulation.

It prints `PASSED` or the number of mismatching vectors per operation, and releases the kernel and device after each step. Missing xclbins are reported and skipped.

In [13]:
for version in VERSIONS:
    print(f"\nChecking Correctness: {version}")
    for variant, params in KYBER_PARAMS.items():
        if variant not in SW_LIBS:
            continue
            
        lib = SW_LIBS[variant]
        device = pyxrt.device(DEVICE_INDEX)
        
        # Keygen Test
        cfg_kg = get_hw_cfg(variant, "keygen", version)
        if not os.path.isfile(cfg_kg["xclbin"]):
            print(f"  {variant} (KEYGEN): File {cfg_kg['xclbin']} not found.")
        else:
            xclbin_obj = pyxrt.xclbin(cfg_kg["xclbin"])
            device.load_xclbin(xclbin_obj)
            uuid = device.get_xclbin_uuid()
            kernel_kg = pyxrt.kernel(device, uuid, cfg_kg["kernel_name"])
            
            errors_kg = 0
            n_checked = min(100, len(KEYGEN_SEEDS))
            
            for i in range(n_checked):
                coins = KEYGEN_SEEDS[i].copy()
                
                expected_pk = np.zeros(params["pk_bytes"], dtype=np.uint8)
                expected_sk = np.zeros(params["sk_bytes"], dtype=np.uint8)
                lib.crypto_kem_keypair_derand(ptr(expected_pk), ptr(expected_sk), ptr(coins))

                hw_pk = np.zeros(params["pk_bytes"], dtype=np.uint8)
                hw_sk = np.zeros(params["sk_bytes"], dtype=np.uint8)

                bo_coins = alloc_bo(device, kernel_kg, coins, 0)
                bo_pk    = alloc_bo(device, kernel_kg, hw_pk, 1)
                bo_sk    = alloc_bo(device, kernel_kg, hw_sk, 2)

                run = kernel_kg(bo_coins, bo_pk, bo_sk) 
                run.wait()

                hw_pk = read_bo(bo_pk, params["pk_bytes"])
                hw_sk = read_bo(bo_sk, params["sk_bytes"])

                if bytes(hw_pk) != bytes(expected_pk) or bytes(hw_sk) != bytes(expected_sk):
                    errors_kg += 1

                del run, bo_coins, bo_pk, bo_sk

            status_kg = "PASSED" if errors_kg == 0 else f"Failed ({errors_kg}/{n_checked} errors)"
            print(f"  {variant} (KEYGEN): {status_kg} ({n_checked} vectors checked)")

            del kernel_kg, xclbin_obj
            gc.collect()

        # Encaps Test
        cfg_enc = get_hw_cfg(variant, "enc", version)
        if not os.path.isfile(cfg_enc["xclbin"]):
            print(f"  {variant} (ENC):    File {cfg_enc['xclbin']} not found.")
        else:
            xclbin_obj = pyxrt.xclbin(cfg_enc["xclbin"])
            device.load_xclbin(xclbin_obj)
            uuid = device.get_xclbin_uuid()
            kernel_enc = pyxrt.kernel(device, uuid, cfg_enc["kernel_name"])
            
            errors_enc = 0
            n_checked = min(100, len(KEYGEN_SEEDS), len(ENC_SEEDS))
            
            for i in range(n_checked):
                inputs = generate_enc_input(lib, params, coins_kg=KEYGEN_SEEDS[i], coins_enc=ENC_SEEDS[i])
                            
                expected_ct = np.zeros(params["ct_bytes"], dtype=np.uint8)
                expected_ss = np.zeros(params["ss_bytes"], dtype=np.uint8)
                lib.crypto_kem_enc_derand(ptr(expected_ct), ptr(expected_ss), ptr(inputs["pk"]), ptr(inputs["coins"]))

                hw_ct = np.zeros(params["ct_bytes"], dtype=np.uint8)
                hw_ss = np.zeros(params["ss_bytes"], dtype=np.uint8)

                bo_pk    = alloc_bo(device, kernel_enc, inputs["pk"], 0)
                bo_coins = alloc_bo(device, kernel_enc, inputs["coins"], 1)
                bo_ct    = alloc_bo(device, kernel_enc, hw_ct, 2)
                bo_ss    = alloc_bo(device, kernel_enc, hw_ss, 3)

                run = kernel_enc(bo_pk, bo_coins, bo_ct, bo_ss)
                run.wait()

                hw_ct = read_bo(bo_ct, params["ct_bytes"])
                hw_ss = read_bo(bo_ss, params["ss_bytes"])

                if bytes(hw_ct) != bytes(expected_ct) or bytes(hw_ss) != bytes(expected_ss):
                    errors_enc += 1

                del run, bo_pk, bo_coins, bo_ct, bo_ss

            status_enc = "PASSED" if errors_enc == 0 else f"Failed ({errors_enc}/{n_checked} errors)"
            print(f"  {variant} (ENC):    {status_enc} ({n_checked} vectors checked)")

            del kernel_enc, xclbin_obj
            gc.collect()

        # Decaps Test
        cfg_dec = get_hw_cfg(variant, "dec", version)
        if not os.path.isfile(cfg_dec["xclbin"]):
            print(f"  {variant} (DEC):    File {cfg_dec['xclbin']} not found.")
        else:
            xclbin_obj = pyxrt.xclbin(cfg_dec["xclbin"])
            device.load_xclbin(xclbin_obj)
            uuid = device.get_xclbin_uuid()
            kernel_dec = pyxrt.kernel(device, uuid, cfg_dec["kernel_name"])
            
            errors_dec = 0
            n_checked = min(100, len(KEYGEN_SEEDS), len(ENC_SEEDS))
            
            for i in range(n_checked):
                inputs = generate_dec_input(lib, params, coins_kg=KEYGEN_SEEDS[i], coins_enc=ENC_SEEDS[i])
                expected_ss = inputs["expected_ss"]

                hw_ss = np.zeros(params["ss_bytes"], dtype=np.uint8)

                bo_ss = alloc_bo(device, kernel_dec, hw_ss, 0)
                bo_ct = alloc_bo(device, kernel_dec, inputs["ct"], 1)
                bo_sk = alloc_bo(device, kernel_dec, inputs["sk"], 2)

                run = kernel_dec(bo_ss, bo_ct, bo_sk)
                run.wait()
                hw_ss = read_bo(bo_ss, params["ss_bytes"])

                if bytes(hw_ss) != bytes(expected_ss):
                    errors_dec += 1

                del run, bo_ct, bo_sk, bo_ss

            status_dec = "PASSED" if errors_dec == 0 else f"Failed ({errors_dec}/{n_checked} errors)"
            print(f"  {variant} (DEC):    {status_dec} ({n_checked} vectors checked)")

            del kernel_dec, xclbin_obj
            gc.collect()

        del device
        gc.collect()


Checking Correctness: v1
  Kyber512 (KEYGEN): PASSED (100 vectors checked)
  Kyber512 (ENC):    PASSED (100 vectors checked)
  Kyber512 (DEC):    PASSED (100 vectors checked)
  Kyber768 (KEYGEN): PASSED (100 vectors checked)
  Kyber768 (ENC):    PASSED (100 vectors checked)
  Kyber768 (DEC):    PASSED (100 vectors checked)
  Kyber1024 (KEYGEN): PASSED (100 vectors checked)
  Kyber1024 (ENC):    PASSED (100 vectors checked)
  Kyber1024 (DEC):    PASSED (100 vectors checked)

Checking Correctness: v2
  Kyber512 (KEYGEN): PASSED (100 vectors checked)
  Kyber512 (ENC):    PASSED (100 vectors checked)
  Kyber512 (DEC):    PASSED (100 vectors checked)
  Kyber768 (KEYGEN): PASSED (100 vectors checked)
  Kyber768 (ENC):    PASSED (100 vectors checked)
  Kyber768 (DEC):    PASSED (100 vectors checked)
  Kyber1024 (KEYGEN): PASSED (100 vectors checked)
  Kyber1024 (ENC):    PASSED (100 vectors checked)
  Kyber1024 (DEC):    PASSED (100 vectors checked)

Checking Correctness: v3
  Kyber512 (KEYG

# Exporting the Results

**Excel export.** The next cell saves the consolidated table to `benchmark_kyber_results.xlsx`, keeping the grouped column headers and the (Variant, Operation, Version) index.

In [14]:
excel_filename = "benchmark_kyber_results.xlsx"
table.to_excel(excel_filename, index=True)

**CSV export.** The next cell writes two CSV files:
1. `benchmark_kyber_results.csv`: the consolidated table above, with the two-level column headers flattened into single names of the form `group | metric` (e.g. `hardware_Time total (ms) | Mean ± CI`) and the Variant / Operation / Version index turned into regular columns. Values keep the same formatting as the displayed table.
2. `benchmark_kyber_raw_results.csv`: the raw, unformatted `combined` DataFrame (one row per platform/version/variant/operation, with numeric values at full precision), which is more convenient for plotting or further analysis.

Both files use the `utf-8-sig` encoding so that symbols like `±` and `μ` display correctly when opened in Excel.

In [15]:
# 1) Consolidated table (same content as the Excel file), with flattened headers
csv_table = table.copy()
csv_table.columns = [f"{group} | {metric}" for group, metric in csv_table.columns]
csv_table = csv_table.reset_index()

csv_filename = "benchmark_kyber_results.csv"
csv_table.to_csv(csv_filename, index=False, encoding="utf-8-sig")
print(f"Consolidated table saved to {csv_filename} ({csv_table.shape[0]} rows x {csv_table.shape[1]} columns)")

# 2) Raw numeric results (HW + SW) at full precision
raw_csv_filename = "benchmark_kyber_raw_results.csv"
combined.to_csv(raw_csv_filename, index=False, encoding="utf-8-sig")
print(f"Raw results saved to {raw_csv_filename} ({combined.shape[0]} rows x {combined.shape[1]} columns)")

Consolidated table saved to benchmark_kyber_results.csv (27 rows x 45 columns)
Raw results saved to benchmark_kyber_raw_results.csv (36 rows x 49 columns)
